In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'

df = pd.read_parquet(RAW / 'EURUSD_M5_recente.parquet')
df = df.sort_values('time').reset_index(drop=True)

print(f"Shape inicial: {df.shape}")
print(f"Período: {df['time'].min()} → {df['time'].max()}")

# ============================================================
# FEATURES DA VELA (forma do candle)
# ============================================================
df['body'] = (df['close'] - df['open']) / df['open']
df['range'] = (df['high'] - df['low']) / df['close']
df['body_abs'] = df['body'].abs()

# Pavios (sombras)
df['upper_shadow'] = (df['high'] - df[['open','close']].max(axis=1)) / df['close']
df['lower_shadow'] = (df[['open','close']].min(axis=1) - df['low']) / df['close']

# Onde o preço fechou dentro do range (0 = fundo, 1 = topo)
df['close_position'] = (df['close'] - df['low']) / (df['high'] - df['low'] + 1e-9)

# Razão corpo / range (marubozu vs doji)
df['body_range_ratio'] = df['body_abs'] / (df['range'] + 1e-9)

# Direção do candle
df['is_bull'] = (df['close'] > df['open']).astype(int)
df['is_bear'] = (df['close'] < df['open']).astype(int)

# Gap da abertura em relação ao candle anterior
df['gap'] = (df['open'] - df['close'].shift(1)) / df['close'].shift(1)

print(f"Após features de vela: {df.shape}")

Shape inicial: (10000, 8)
Período: 2026-08-10 06:40:00 → 2026-09-25 23:55:00
Após features de vela: (10000, 18)


In [2]:
# Doji: corpo muito pequeno
df['doji'] = (df['body_range_ratio'] < 0.1).astype(int)

# Hammer: pavio inferior longo, corpo no topo
df['hammer'] = (
    (df['lower_shadow'] > 2 * df['body_abs']) &
    (df['upper_shadow'] < df['body_abs']) &
    (df['is_bull'] == 1)
).astype(int)

# Inverted Hammer
df['inv_hammer'] = (
    (df['upper_shadow'] > 2 * df['body_abs']) &
    (df['lower_shadow'] < df['body_abs']) &
    (df['is_bear'] == 1)
).astype(int)

# Marubozu: sem pavios, corpo grande
df['marubozu'] = (
    (df['upper_shadow'] < 0.1 * df['body_abs']) &
    (df['lower_shadow'] < 0.1 * df['body_abs']) &
    (df['body_abs'] > 0.001)
).astype(int)

# Engulfing (bull): candle bull engole candle bear anterior
df['engulfing_bull'] = (
    (df['is_bull'] == 1) &
    (df['is_bear'].shift(1) == 1) &
    (df['close'] > df['open'].shift(1)) &
    (df['open'] < df['close'].shift(1))
).astype(int)

# Engulfing (bear)
df['engulfing_bear'] = (
    (df['is_bear'] == 1) &
    (df['is_bull'].shift(1) == 1) &
    (df['close'] < df['open'].shift(1)) &
    (df['open'] > df['close'].shift(1))
).astype(int)

# Soma quantos padrões ativaram
df['n_patterns'] = df[['doji','hammer','inv_hammer','marubozu','engulfing_bull','engulfing_bear']].sum(axis=1)

print(f"Após padrões: {df.shape}")
print(f"Frequência de padrões:")
print(df[['doji','hammer','inv_hammer','marubozu','engulfing_bull','engulfing_bear']].mean().round(3))

Após padrões: (10000, 25)
Frequência de padrões:
doji              0.113
hammer            0.023
inv_hammer        0.024
marubozu          0.000
engulfing_bull    0.009
engulfing_bear    0.010
dtype: float64


In [3]:
# Retornos dos últimos N candles
for lag in [1, 2, 3, 5, 10]:
    df[f'ret_lag{lag}'] = df['close'].pct_change(lag)

# Volatilidade rolling
df['vol_ret_10'] = df['close'].pct_change().rolling(10).std()
df['vol_ret_20'] = df['close'].pct_change().rolling(20).std()
df['vol_ratio'] = df['vol_ret_10'] / (df['vol_ret_20'] + 1e-9)

# Médias móveis
for w in [5, 10, 20, 50]:
    df[f'mm{w}'] = df['close'].rolling(w).mean()
    df[f'dist_mm{w}'] = (df['close'] / df[f'mm{w}']) - 1

# Cruzamento de MMs
df['mm_cross'] = (df['mm5'] / df['mm10']) - 1

# RSI (14 candles)
def rsi(series, window=14):
    delta = series.diff()
    ganho = delta.clip(lower=0).rolling(window).mean()
    perda = (-delta.clip(upper=0)).rolling(window).mean()
    return 100 - (100 / (1 + ganho / (perda + 1e-9)))

df['rsi14'] = rsi(df['close'], 14)
df['rsi_centered'] = df['rsi14'] - 50  # 0 = neutro

# Volume relativo
df['vol_ma20'] = df['tick_volume'].rolling(20).mean()
df['vol_rel'] = df['tick_volume'] / (df['vol_ma20'] + 1e-9)
df['log_vol'] = np.log1p(df['tick_volume'])

# Bollinger Bands
df['bb_mid'] = df['close'].rolling(20).mean()
df['bb_std'] = df['close'].rolling(20).std()
df['bb_pos'] = (df['close'] - df['bb_mid']) / (2 * df['bb_std'] + 1e-9)

# Hora do dia (importante em forex — sessões)
df['hora'] = df['time'].dt.hour
df['minuto'] = df['time'].dt.minute
df['dia_semana'] = df['time'].dt.dayofweek

print(f"Após contexto: {df.shape}")

Após contexto: (10000, 53)


In [4]:
# Target de regressão: retorno do próximo candle
df['target'] = df['close'].pct_change().shift(-1)

# Target binário: o próximo candle sobe?
df['target_bin'] = (df['target'] > 0).astype(int)

# Target com threshold (movimento mínimo para valer a pena)
# Ex: só considera "compra" se o próximo candle subir > 0.05%
THRESHOLD = 0.0002
df['target_bin_thresh'] = (df['target'] > THRESHOLD).astype(int)
df['target_bin_thresh'] = df['target_bin_thresh'].where(df['target'].notna())

print(f"Target (binário): {df['target_bin'].value_counts(normalize=True).to_dict()}")
print(f"Target (thresh {THRESHOLD}): {df['target_bin_thresh'].value_counts(normalize=True).to_dict()}")

Target (binário): {0: 0.5242, 1: 0.4758}
Target (thresh 0.0002): {0.0: 0.933993399339934, 1.0: 0.066006600660066}


In [5]:
FEATURES = [
    # Vela
    'body', 'range', 'body_abs', 'upper_shadow', 'lower_shadow',
    'close_position', 'body_range_ratio', 'is_bull', 'is_bear', 'gap',
    # Padrões
    'doji', 'hammer', 'inv_hammer', 'marubozu', 'engulfing_bull', 'engulfing_bear', 'n_patterns',
    # Contexto
    'ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_lag5', 'ret_lag10',
    'vol_ret_10', 'vol_ret_20', 'vol_ratio',
    'dist_mm5', 'dist_mm10', 'dist_mm20', 'dist_mm50', 'mm_cross',
    'rsi_centered', 'bb_pos',
    # Volume
    'vol_rel', 'log_vol',
    # Tempo (categóricas — vão virar dummies)
    # 'hora',  (opcional, ver abaixo)
]

# Se quiser incluir hora como dummy
hora_dummies = pd.get_dummies(df['hora'], prefix='h', drop_first=True).astype(int)
df = pd.concat([df, hora_dummies], axis=1)
FEATURES += list(hora_dummies.columns)

print(f"Total de features: {len(FEATURES)}")

Total de features: 57


In [6]:
# Remove NaN gerados pelos rolling
df_model = df.dropna(subset=['target_bin'] + FEATURES).copy()

# Adiciona colunas auxiliares
df_model = df_model[['time', 'open', 'high', 'low', 'close', 'tick_volume',
                      'target', 'target_bin', 'target_bin_thresh'] + FEATURES].copy()

print(f"Shape final: {df_model.shape}")
print(f"Período: {df_model['time'].min()} → {df_model['time'].max()}")

output = RAW / 'EURUSD_M5_features.parquet'
df_model.to_parquet(output, index=False)
print(f"✅ Salvo: {output}")

Shape final: (9951, 66)
Período: 2026-08-10 10:45:00 → 2026-09-25 23:55:00
✅ Salvo: c:\Users\rondy\Desenvolvimento\b3-ml\data\raw\EURUSD_M5_features.parquet


In [5]:
# Rastreia a definição atual
import numpy as np

# Testa manualmente as 3 condições
c1 = df['upper_shadow'] < 0.1 * df['body_abs']
c2 = df['lower_shadow'] < 0.1 * df['body_abs']
c3 = df['body_abs'] > 0.001

print(f"c1 (upper < 0.1*body): {c1.sum()}")
print(f"c2 (lower < 0.1*body): {c2.sum()}")
print(f"c3 (body > 0.001):     {c3.sum()}")
print(f"c1 & c2 & c3:          {(c1 & c2 & c3).sum()}")
print(f"\nFrequência final: {(c1 & c2 & c3).mean():.4f}")

c1 (upper < 0.1*body): 1845
c2 (lower < 0.1*body): 2117
c3 (body > 0.001):     14
c1 & c2 & c3:          3

Frequência final: 0.0003


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = ROOT / 'data' / 'raw'

# Carrega limpo
df = pd.read_parquet(RAW / 'EURUSD_M5_recente.parquet')
df = df.sort_values('time').reset_index(drop=True)

# ---------- Features da vela ----------
df['body'] = (df['close'] - df['open']) / df['open']
df['range'] = (df['high'] - df['low']) / df['close']
df['body_abs'] = df['body'].abs()
df['upper_shadow'] = (df['high'] - df[['open','close']].max(axis=1)) / df['close']
df['lower_shadow'] = (df[['open','close']].min(axis=1) - df['low']) / df['close']
df['close_position'] = (df['close'] - df['low']) / (df['high'] - df['low'] + 1e-9)
df['body_range_ratio'] = df['body_abs'] / (df['range'] + 1e-9)
df['is_bull'] = (df['close'] > df['open']).astype(int)
df['is_bear'] = (df['close'] < df['open']).astype(int)
df['gap'] = (df['open'] - df['close'].shift(1)) / df['close'].shift(1)

# ---------- Padrões (SEM marubozu) ----------
df['doji'] = (df['body_range_ratio'] < 0.1).astype(int)
df['hammer'] = (
    (df['lower_shadow'] > 2 * df['body_abs']) &
    (df['upper_shadow'] < df['body_abs']) &
    (df['is_bull'] == 1)
).astype(int)
df['inv_hammer'] = (
    (df['upper_shadow'] > 2 * df['body_abs']) &
    (df['lower_shadow'] < df['body_abs']) &
    (df['is_bear'] == 1)
).astype(int)
df['engulfing_bull'] = (
    (df['is_bull'] == 1) &
    (df['is_bear'].shift(1) == 1) &
    (df['close'] > df['open'].shift(1)) &
    (df['open'] < df['close'].shift(1))
).astype(int)
df['engulfing_bear'] = (
    (df['is_bear'] == 1) &
    (df['is_bull'].shift(1) == 1) &
    (df['close'] < df['open'].shift(1)) &
    (df['open'] > df['close'].shift(1))
).astype(int)

# ---------- Contexto ----------
for lag in [1, 2, 3, 5, 10]:
    df[f'ret_lag{lag}'] = df['close'].pct_change(lag)

df['vol_ret_10'] = df['close'].pct_change().rolling(10).std()
df['vol_ret_20'] = df['close'].pct_change().rolling(20).std()
df['vol_ratio'] = df['vol_ret_10'] / (df['vol_ret_20'] + 1e-9)

for w in [5, 10, 20, 50]:
    df[f'mm{w}'] = df['close'].rolling(w).mean()
    df[f'dist_mm{w}'] = (df['close'] / df[f'mm{w}']) - 1
df['mm_cross'] = (df['mm5'] / df['mm10']) - 1

def rsi(series, window=14):
    delta = series.diff()
    ganho = delta.clip(lower=0).rolling(window).mean()
    perda = (-delta.clip(upper=0)).rolling(window).mean()
    return 100 - (100 / (1 + ganho / (perda + 1e-9)))

df['rsi14'] = rsi(df['close'], 14)
df['rsi_centered'] = df['rsi14'] - 50

df['vol_ma20'] = df['tick_volume'].rolling(20).mean()
df['vol_rel'] = df['tick_volume'] / (df['vol_ma20'] + 1e-9)
df['log_vol'] = np.log1p(df['tick_volume'])

df['bb_mid'] = df['close'].rolling(20).mean()
df['bb_std'] = df['close'].rolling(20).std()
df['bb_pos'] = (df['close'] - df['bb_mid']) / (2 * df['bb_std'] + 1e-9)

# ---------- Hora (dummies) ----------
df['hora'] = df['time'].dt.hour
hora_dummies = pd.get_dummies(df['hora'], prefix='h', drop_first=True).astype(int)
df = pd.concat([df, hora_dummies], axis=1)

# ---------- Target ----------
df['target'] = df['close'].pct_change().shift(-1)
df['target_bin'] = (df['target'] > 0).astype(int)

# ---------- Features ----------
FEATURES = [
    'body', 'range', 'body_abs', 'upper_shadow', 'lower_shadow',
    'close_position', 'body_range_ratio', 'is_bull', 'is_bear', 'gap',
    'doji', 'hammer', 'inv_hammer', 'engulfing_bull', 'engulfing_bear',
    'ret_lag1', 'ret_lag2', 'ret_lag3', 'ret_lag5', 'ret_lag10',
    'vol_ret_10', 'vol_ret_20', 'vol_ratio',
    'dist_mm5', 'dist_mm10', 'dist_mm20', 'dist_mm50', 'mm_cross',
    'rsi_centered', 'bb_pos',
    'vol_rel', 'log_vol',
] + list(hora_dummies.columns)

# ---------- Dataset final ----------
df_model = df.dropna(subset=['target_bin'] + FEATURES).copy()
df_model = df_model[['time','open','high','low','close','tick_volume',
                      'target','target_bin'] + FEATURES].copy()

print(f"✅ Shape final: {df_model.shape}")
print(f"   Features: {len(FEATURES)}")
print(f"   Target: {df_model['target_bin'].value_counts(normalize=True).to_dict()}")
print(f"\n   Frequência dos padrões:")
for p in ['doji','hammer','inv_hammer','engulfing_bull','engulfing_bear']:
    print(f"     {p:20s} {df_model[p].mean():.4f}")

# Salva
output = RAW / 'EURUSD_M5_features.parquet'
df_model.to_parquet(output, index=False)
print(f"\n✅ Salvo: {output}")

✅ Shape final: (9951, 63)
   Features: 55
   Target: {0: 0.5242689176967139, 1: 0.4757310823032861}

   Frequência dos padrões:
     doji                 0.1125
     hammer               0.0228
     inv_hammer           0.0237
     engulfing_bull       0.0086
     engulfing_bear       0.0096

✅ Salvo: c:\Users\rondy\Desenvolvimento\b3-ml\data\raw\EURUSD_M5_features.parquet
